# 🚀 Meeting-to-Task Agent Workflow Demo

This notebook demonstrates the **Meeting-to-Task Agent** workflow:

```
📹 Meeting Recording
    ↓
🎙️ Speech-to-Text (STT)
    ↓
📊 Meeting Analysis (Summary + Tasks)
    ↓
✅ Quality Check (Self-Review)
    ↓
🔄 Refinement (if needed)
    ↓
⏸️ **PAUSE FOR HUMAN REVIEW**
    ↓
📝 Create Tasks in System
    ↓
📧 Send Notifications
```

---

## Step 1: Configuration & Setup

Load environment variables and configure the API endpoint.

In [17]:
import requests
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import uuid

# Load environment variables from .env
load_dotenv(dotenv_path="../../.env")

# API Configuration
BASE_URL = "http://localhost:8001/api/v1"
MEETING_API_URL = f"{BASE_URL}/meeting"

# Get API Bearer Token from .env (for creating tasks)
API_BEARER_TOKEN = os.getenv("API_BEARER_TOKEN")

HEADERS = {
    "Content-Type": "application/json",
}

if API_BEARER_TOKEN:
    HEADERS["Authorization"] = f"Bearer {API_BEARER_TOKEN}"
    print("✅ API Bearer Token loaded successfully")
else:
    print("⚠️  Warning: API_BEARER_TOKEN not found in .env file")

print(f"\n🌐 API Endpoint: {MEETING_API_URL}")

✅ API Bearer Token loaded successfully

🌐 API Endpoint: http://localhost:8001/api/v1/meeting


## Step 2: Prepare Meeting Data

Set up the meeting metadata and verify the video file exists.

In [18]:
# Meeting metadata
meeting_metadata = {
    "title": "Aentic Application",
    "description": "Building agentic ai application",
    "project_id": "484a97a2-8fbf-4da3-a42f-ed89bf3d73db",
    "author_id": "52de8e7c-40d3-488f-8888-85d9aaa005eb",
    "participants": [
        {"id": "52de8e7c-40d3-488f-8888-85d9aaa005eb", "name": "Hùng", "email": "nguyenquangphuoc9614@gmail.com"},
        {"id": "754f0dad-3d6d-47d8-bdf7-f9d30496f2f7", "name": "Tuấn", "email": "nguyenquangphuoc962004@gmail.com"},
        {"id": "b05deb68-3629-4c68-adb4-8d7dba8034e7", "name": "Lan", "email": "chauthanhdat174@gmail.com"},
        {"id": "dfb297da-7b37-4650-bbde-fb1adaff59be", "name": "Nam", "email": "zzvandatzz9@gmail.com"}
    ]
}

# Audio/Video file path
NOTEBOOK_DIR = Path(os.getcwd())
MEETING_VIDEO_PATH = str(NOTEBOOK_DIR / "meeting001.webm")

# Generate unique meeting ID
MEETING_ID = str(uuid.uuid4())

# Verify file exists
if os.path.exists(MEETING_VIDEO_PATH):
    file_size = os.path.getsize(MEETING_VIDEO_PATH) / (1024*1024)
    print(f"📹 Meeting Recording: {Path(MEETING_VIDEO_PATH).name}")
    print(f"   Size: {file_size:.2f} MB")
else:
    print(f"❌ ERROR: Video file not found at {MEETING_VIDEO_PATH}")

print(f"\n🆔 Meeting ID: {MEETING_ID}")
print(f"👥 Participants: {len(meeting_metadata['participants'])} people")

📹 Meeting Recording: meeting001.webm
   Size: 1.75 MB

🆔 Meeting ID: 1c404672-cc5a-4ddf-8791-70e64fa6a0bf
👥 Participants: 4 people


## Step 3: Start the Agent Workflow

Call the `/analyze` endpoint with `skip_review=False` to pause before task creation.

The agent will:
1. 🎙️ **Transcribe** the audio to text
2. 📊 **Analyze** the meeting (create summary & extract tasks)
3. ✅ **Quality Check** using self-reflection
4. 🔄 **Refine** if quality check suggests improvements
5. ⏸️ **Pause** and wait for human review

**Note:** Check your **API server console** to see detailed logs of each workflow node!

In [19]:
# Prepare request
analyze_request = {
    "meeting_id": MEETING_ID,
    "title": meeting_metadata["title"],
    "description": meeting_metadata["description"],
    "author_id": meeting_metadata["author_id"],
    "project_id": meeting_metadata["project_id"],
    "audio_file_path": MEETING_VIDEO_PATH,
    "participants": meeting_metadata["participants"],
}

print("🚀 Starting Meeting-to-Task Agent...")
print("📊 Processing workflow: STT → Analysis → Quality Check → Review\n")
print("⏳ This may take a few minutes...")
print("💡 TIP: Watch the API server console for detailed workflow logs!\n")

# Call API
response = requests.post(
    f"{MEETING_API_URL}/analyze",
    headers=HEADERS,
    json=analyze_request,
    params={
        "background": False,   # Synchronous execution
        "skip_review": False   # Pause for human review
    },
    timeout=600
)

if response.status_code == 200:
    result = response.json()
    print("\n" + "="*70)
    print("✅ WORKFLOW PAUSED - READY FOR REVIEW")
    print("="*70)
    print(f"\n📊 Status: {result.get('status')}")
    print(f"🆔 Thread ID: {result.get('thread_id')}")
else:
    print(f"\n❌ Error: {response.status_code}")
    print(response.text)

🚀 Starting Meeting-to-Task Agent...
📊 Processing workflow: STT → Analysis → Quality Check → Review

⏳ This may take a few minutes...
💡 TIP: Watch the API server console for detailed workflow logs!


✅ WORKFLOW PAUSED - READY FOR REVIEW

📊 Status: waiting_review
🆔 Thread ID: 1c404672-cc5a-4ddf-8791-70e64fa6a0bf


## Step 4: Review the Analysis Results

Let's examine what the agent extracted from the meeting.

In [20]:
if response.status_code == 200:
    result = response.json()
    
    # Display summary
    print("\n" + "="*70)
    print("📝 MEETING SUMMARY")
    print("="*70)
    print(result.get('summary', 'No summary available'))
    
    # Display action items
    print("\n" + "="*70)
    print(f"📋 ACTION ITEMS ({len(result.get('action_items', []))})")
    print("="*70)
    
    for i, item in enumerate(result.get('action_items', []), 1):
        print(f"\n{i}. {item.get('title')}")
        print(f"   👤 Assignee: {item.get('assignee')}")
        print(f"   📅 Due: {item.get('due_date')}")
        print(f"   🎯 Priority: {item.get('priority')}")
        if item.get('description'):
            print(f"   📄 {item.get('description')}")
    
    # Display transcript preview
    print("\n" + "="*70)
    print("🎤 TRANSCRIPT PREVIEW")
    print("="*70)
    transcript = result.get('transcript', '')
    preview_length = 500
    if len(transcript) > preview_length:
        print(transcript[:preview_length] + "...")
    else:
        print(transcript)
else:
    print("❌ Cannot display results - API call failed")


📝 MEETING SUMMARY
Mục tiêu: Cuộc họp nhằm chốt nhanh danh sách các task trên hệ thống cho sprint 42, điền đầy đủ thông tin cần thiết và đẩy vào database.Thảo luận chính:<ul><li>Xác định chi tiết các task cần thực hiện, bao gồm tiêu đề, mô tả và độ ưu tiên.</li><li>Ước lượng độ khó (points) và thời gian thực hiện cho từng task.</li><li>Phân công người chịu trách nhiệm cho từng task.</li><li>Xác định deadline và trạng thái ban đầu của các task.</li></ul>Quyết định:<ul><li>Task "Tích hợp cổng thanh toán VNpay" sẽ được giao cho Tuấn, ưu tiên High, hoàn thành trước ngày 10 tháng 12.</li><li>Task "Thiết kế lại giao diện dashboard báo cáo" sẽ được giao cho Lan, ưu tiên Medium, hoàn thành trước ngày 12 tháng 12.</li><li>Task "Fix lỗi crash app khi upload ảnh quá 5MB" sẽ được giao cho Nam, ưu tiên High, hoàn thành trong ngày 03 tháng 12 và được bắt đầu ngay lập tức.</li><li>Tất cả các task đều thuộc dự án "Super App V2".</li></ul>

📋 ACTION ITEMS (3)

1. Tích hợp cổng thanh toán VNpay
   👤 Ass

## Step 5: Continue the Workflow (Create Tasks & Send Notifications)

Now that we've reviewed the results, let's continue the workflow to:
1. 📝 **Create tasks** in the project management system
2. 📧 **Send email notifications** to assignees

You can modify the summary or action items before confirming if needed.

**Note:** Watch the API server console to see:
- Task creation status
- Email notification details

In [21]:
if response.status_code == 200 and result.get('status') == 'waiting_review':
    
    # Use original results (or you can modify them here)
    updated_summary = result.get('summary')
    updated_action_items = result.get('action_items', [])
    
    # Example: Modify action items if needed
    # if updated_action_items:
    #     updated_action_items[0]['priority'] = 'High'
    
    confirm_request = {
        "meeting_id": MEETING_ID,
        "updated_summary": updated_summary,
        "updated_action_items": updated_action_items,
        "project_id": meeting_metadata["project_id"],
        "author_id": meeting_metadata["author_id"],
        "participants": meeting_metadata["participants"]
    }
    
    print("🚀 Continuing workflow...")
    print("📊 Processing: Task Creation → Notifications\n")
    print("💡 TIP: Watch the API server console for detailed logs!\n")
    
    # Confirm and continue
    confirm_response = requests.post(
        f"{MEETING_API_URL}/confirm",
        headers=HEADERS,
        json=confirm_request,
        timeout=120
    )
    
    if confirm_response.status_code == 200:
        confirm_result = confirm_response.json()
        
        print("\n" + "="*70)
        print("🎉 WORKFLOW COMPLETED SUCCESSFULLY!")
        print("="*70)
        print(f"\n📊 Status: {confirm_result.get('status')}")
        print(f"📝 Tasks Created: {len(confirm_result.get('action_items', []))}")
        
        # Show what was created
        print("\n✅ The following tasks were created:")
        for i, item in enumerate(confirm_result.get('action_items', []), 1):
            print(f"   {i}. {item.get('title')}")
        
        print("\n📧 Email notifications have been sent to assignees.")
        print("\n💡 Check the API server console for detailed task creation and notification logs.")
        
    else:
        print(f"\n❌ Error: {confirm_response.status_code}")
        print(confirm_response.text)
else:
    print("⚠️  Skipping - either API failed or status is not 'waiting_review'")

🚀 Continuing workflow...
📊 Processing: Task Creation → Notifications

💡 TIP: Watch the API server console for detailed logs!


🎉 WORKFLOW COMPLETED SUCCESSFULLY!

📊 Status: completed
📝 Tasks Created: 3

✅ The following tasks were created:
   1. Tích hợp cổng thanh toán VNpay
   2. Thiết kế lại giao diện dashboard báo cáo
   3. Fix lỗi crash app khi upload ảnh quá 5MB

📧 Email notifications have been sent to assignees.

💡 Check the API server console for detailed task creation and notification logs.


## 🎯 Workflow Summary

The Meeting-to-Task Agent has successfully:

1. ✅ **Transcribed** the meeting audio to text
2. ✅ **Analyzed** the meeting and extracted key information
3. ✅ **Quality checked** the results using self-reflection
4. ✅ **Paused** for human review (you had a chance to modify results)
5. ✅ **Created tasks** in your project management system
6. ✅ **Sent notifications** to task assignees via email

### 📊 What Happened Behind the Scenes

Check your **API server console** to see detailed logs showing:

- **[NODE 1] Speech-to-Text**: Transcript length and preview
- **[NODE 2] Meeting Analysis**: Full summary and tasks (pretty-printed)
- **[NODE 3] Quality Check**: Decision and critique from self-review
- **[NODE 4] Refinement**: Updated summary/tasks (if quality check suggested improvements)
- **[NODE 5] Create Tasks**: Success status and number of tasks created
- **[NODE 6] Send Notifications**: Number of emails sent and recipient list

---

🎊 **Congratulations!** You've successfully automated meeting-to-task conversion with AI!